# Simmetrie residue e annullamento strutturale dei correlatori — trimero a catena aperta

Mirror computazionale di `simmetrie_correlatori_trimero_catena.tex`, per la topologia a
catena aperta (2 legami). Verifica passo per passo, non solo teoria, di ogni affermazione
del documento — stessa impostazione del mirror dell'anello
(`correlazioni_trimero_anello_simmetria.ipynb`).

Punto di lavoro primario: $J=1, b=b_c=3.0, D=0.15$ (VQE-DM).

## 0. Setup

In [1]:
import numpy as np
import scipy.linalg as sla
from qiskit.quantum_info import Pauli

from trimer_chain_exact import trimer_hamiltonian_dm

pos = {1: 0, 2: 1, 3: 2}

def site_op(site, alpha):
    lab = ["I", "I", "I"]
    lab[pos[site]] = alpha.upper()
    return Pauli("".join(lab)).to_matrix()

J, b, D = 1.0, 3.0, 0.15
H = trimer_hamiltonian_dm(J, b, D).to_matrix()
E, V = np.linalg.eigh(H)
psi0 = V[:, 0]
imax = np.argmax(np.abs(psi0))
psi0 = psi0 * np.exp(-1j * np.angle(psi0[imax]))

print(f"Punto di lavoro: J={J}, b={b}, D={D}")
print(f"E0={E[0]:.6f}  gap(E1-E0)={E[1]-E[0]:.6f}  (non degenere: gap>0)")
print(f"max|Im psi0| = {np.max(np.abs(psi0.imag)):.2e}  (atteso 0: fase reale)")

Punto di lavoro: J=1.0, b=3.0, D=0.15
E0=-7.369247  gap(E1-E0)=0.734742  (non degenere: gap>0)
max|Im psi0| = 0.00e+00  (atteso 0: fase reale)


## 1. $S_z^{tot}$ non è conservato per $D\neq0$ (identico all'anello nella dimostrazione)

In [2]:
Sz_tot = site_op(1, "z") + site_op(2, "z") + site_op(3, "z")
comm_D0 = np.linalg.norm(Sz_tot @ trimer_hamiltonian_dm(J, b, 0.0).to_matrix()
                          - trimer_hamiltonian_dm(J, b, 0.0).to_matrix() @ Sz_tot)
comm_D = np.linalg.norm(Sz_tot @ H - H @ Sz_tot)
print(f"||[Sz_tot,H]|| a D=0    : {comm_D0:.2e}  (atteso 0: conservato)")
print(f"||[Sz_tot,H]|| a D={D}: {comm_D:.4f}  (atteso !=0: rotto dal DM)")

||[Sz_tot,H]|| a D=0    : 0.00e+00  (atteso 0: conservato)
||[Sz_tot,H]|| a D=0.15: 1.6971  (atteso !=0: rotto dal DM)


## 2. $P_{13}$ (SWAP puro) già funziona — a differenza dell'anello

Nell'anello lo scambio puro $P_{12}$ **non basta** con il DM acceso: serve la rotazione
aggiuntiva $R_z(\pi)^{\otimes3}$. Qui verifichiamo che per la catena il puro scambio
$P_{13}$ (siti 1,3, sito 2 fisso), **senza alcuna rotazione**, commuta già con $H$ completo
— grazie al segno $D_{12}=-D_{23}=D$ fissato dalla topologia stessa (nessun legame (3,1)
esiste, quindi nessuna ambiguità di opzione come nell'anello).

In [3]:
def swap_p13_matrix():
    dim = 8
    M = np.zeros((dim, dim))
    for idx in range(dim):
        bits = [(idx >> k) & 1 for k in range(3)]
        bits[0], bits[2] = bits[2], bits[0]
        new_idx = sum(bit << k for k, bit in enumerate(bits))
        M[new_idx, idx] = 1.0
    return M

P13 = swap_p13_matrix()
print("P13 unitaria e involutoria:", np.allclose(P13 @ P13, np.eye(8)),
      np.allclose(P13.conj().T @ P13, np.eye(8)))

comm_ok = np.linalg.norm(P13 @ H - H @ P13)
print(f"\n||[P13,H]|| col segno corretto (D12=-D23=D): {comm_ok:.2e}  (atteso 0)")

# segno "sbagliato": D12=D23=D
H_wrong = trimer_hamiltonian_dm(J, b, 0.0).to_matrix()
d12 = D * (site_op(1, 'x') @ site_op(2, 'z') - site_op(1, 'z') @ site_op(2, 'x'))
d23_wrong = D * (site_op(2, 'x') @ site_op(3, 'z') - site_op(2, 'z') @ site_op(3, 'x'))
H_wrong = H_wrong + d12 + d23_wrong
comm_wrong = np.linalg.norm(P13 @ H_wrong - H_wrong @ P13)
print(f"||[P13,H]|| col segno SBAGLIATO (D12=D23=D): {comm_wrong:.4f}  (atteso grande: rottura netta)")

P13 unitaria e involutoria: True True

||[P13,H]|| col segno corretto (D12=-D23=D): 0.00e+00  (atteso 0)
||[P13,H]|| col segno SBAGLIATO (D12=D23=D): 1.6971  (atteso grande: rottura netta)


## 3. Lemma: azione di $P_{13}$ su un operatore di singolo sito

In [4]:
pi_map = {1: 3, 2: 2, 3: 1}
print("P13 sigma_i^alpha P13^-1 = lambda_alpha * sigma_{pi(i)}^alpha,  pi=(1 3)\n")
for site in (1, 2, 3):
    for alpha in "xyz":
        op = site_op(site, alpha)
        transformed = P13 @ op @ P13.conj().T
        target = site_op(pi_map[site], alpha)
        ratio = None
        for r in range(8):
            for c in range(8):
                if abs(target[r, c]) > 1e-9:
                    ratio = transformed[r, c] / target[r, c]
                    break
            if ratio is not None:
                break
        err = np.linalg.norm(transformed - ratio * target)
        print(f"  site{site}^{alpha} -> site{pi_map[site]}^{alpha} * lambda={ratio.real:+.3f}  (residuo {err:.1e})")
print("\n-> lambda_alpha = +1 per OGNI alpha (puro scambio, nessun segno):")
print("   qualitativamente diverso dall'anello (eta_x=eta_y=-1, eta_z=+1)")

P13 sigma_i^alpha P13^-1 = lambda_alpha * sigma_{pi(i)}^alpha,  pi=(1 3)

  site1^x -> site3^x * lambda=+1.000  (residuo 0.0e+00)
  site1^y -> site3^y * lambda=+1.000  (residuo 0.0e+00)
  site1^z -> site3^z * lambda=+1.000  (residuo 0.0e+00)
  site2^x -> site2^x * lambda=+1.000  (residuo 0.0e+00)
  site2^y -> site2^y * lambda=+1.000  (residuo 0.0e+00)
  site2^z -> site2^z * lambda=+1.000  (residuo 0.0e+00)
  site3^x -> site1^x * lambda=+1.000  (residuo 0.0e+00)
  site3^y -> site1^y * lambda=+1.000  (residuo 0.0e+00)
  site3^z -> site1^z * lambda=+1.000  (residuo 0.0e+00)

-> lambda_alpha = +1 per OGNI alpha (puro scambio, nessun segno):
   qualitativamente diverso dall'anello (eta_x=eta_y=-1, eta_z=+1)


## 4. Corollario: nessuno zero forzato sul sito fisso

Nell'anello il sito fisso (3) produceva 4 zeri strutturali per ogni $t$. Qui verifichiamo
che il sito fisso (2) non è vincolato: $P_{13}$ produce solo **uguaglianze** fra correlatori
distinti ($C_{11}\equiv C_{33}$, etc.), mai zeri, perché $\lambda_\alpha\lambda_\beta=+1$
sempre.

In [5]:
def classical_exact(i, alpha, j, beta, t, H, psi0):
    Ut = sla.expm(-1j * H * t)
    return complex(np.vdot(psi0, Ut.conj().T @ site_op(i, alpha) @ Ut @ site_op(j, beta) @ psi0))

t_test = 1.3
paulis = "xyz"
print("Verifica C_11 = C_33, C_12 = C_32, C_21 = C_23, C_13 = C_31 (tutte le 9 combinazioni ciascuna):\n")
for (pair1, pair2) in [((1, 1), (3, 3)), ((1, 2), (3, 2)), ((2, 1), (2, 3)), ((1, 3), (3, 1))]:
    maxerr = 0.0
    for a in paulis:
        for b_ in paulis:
            c1 = classical_exact(pair1[0], a, pair1[1], b_, t_test, H, psi0)
            c2 = classical_exact(pair2[0], a, pair2[1], b_, t_test, H, psi0)
            maxerr = max(maxerr, abs(c1 - c2))
    print(f"  C_{pair1[0]}{pair1[1]} == C_{pair2[0]}{pair2[1]}: max errore = {maxerr:.2e}")

print("\nC_22 (sito fisso): nessuno zero atteso -- controllo che sia genericamente non nullo")
for a in paulis:
    for b_ in paulis:
        c22 = classical_exact(2, a, 2, b_, t_test, H, psi0)
        print(f"  C_22^{a}{b_}(t={t_test}) = {c22.real:+.4f}{c22.imag:+.4f}j")

Verifica C_11 = C_33, C_12 = C_32, C_21 = C_23, C_13 = C_31 (tutte le 9 combinazioni ciascuna):

  C_11 == C_33: max errore = 4.48e-16
  C_12 == C_32: max errore = 5.03e-16
  C_21 == C_23: max errore = 3.61e-16
  C_13 == C_31: max errore = 5.24e-16

C_22 (sito fisso): nessuno zero atteso -- controllo che sia genericamente non nullo
  C_22^xx(t=1.3) = +0.4047-0.1349j
  C_22^xy(t=1.3) = -0.1209+0.2512j
  C_22^xz(t=1.3) = +0.4190+0.2170j
  C_22^yx(t=1.3) = +0.1209-0.2512j
  C_22^yy(t=1.3) = +0.1426-0.6676j
  C_22^yz(t=1.3) = -0.6585-0.1709j
  C_22^zx(t=1.3) = +0.4190+0.2170j
  C_22^zy(t=1.3) = +0.6585+0.1709j
  C_22^zz(t=1.3) = +0.1916-0.7559j


## 5. Time-reversal $K$: zeri a $t=0$ (due corollari, non uno solo)

In [6]:
print(f"max|Im H| = {np.max(np.abs(H.imag)):.2e}  (atteso 0: H reale)\n")

print("Corollario A (i != j, esattamente una y): 24 zeri attesi")
maxerr = 0.0
n = 0
for i in (1, 2, 3):
    for j in (1, 2, 3):
        if i == j:
            continue
        for a in paulis:
            for b_ in paulis:
                if (a == 'y') + (b_ == 'y') == 1:
                    c0 = classical_exact(i, a, j, b_, 0.0, H, psi0)
                    maxerr = max(maxerr, abs(c0))
                    n += 1
print(f"  {n} combinazioni, max|C(0)| = {maxerr:.2e}\n")

print("Corollario B (i == j, xz/zx): 6 zeri attesi, argomento DIVERSO (<sigma_i^y>=0 per stato reale)")
maxerr = 0.0
for i in (1, 2, 3):
    sy = np.vdot(psi0, site_op(i, 'y') @ psi0)
    for (a, b_) in [('x', 'z'), ('z', 'x')]:
        c0 = classical_exact(i, a, i, b_, 0.0, H, psi0)
        maxerr = max(maxerr, abs(c0))
    print(f"  sito {i}: <sigma^y> = {sy:.2e}")
print(f"  max|C_ii^xz(0)|, |C_ii^zx(0)| su 3 siti = {maxerr:.2e}")
print("\nTotale zeri a t=0 attesi: 24 + 6 = 30")

max|Im H| = 0.00e+00  (atteso 0: H reale)

Corollario A (i != j, esattamente una y): 24 zeri attesi
  24 combinazioni, max|C(0)| = 0.00e+00

Corollario B (i == j, xz/zx): 6 zeri attesi, argomento DIVERSO (<sigma_i^y>=0 per stato reale)
  sito 1: <sigma^y> = 0.00e+00+0.00e+00j
  sito 2: <sigma^y> = 0.00e+00+0.00e+00j
  sito 3: <sigma^y> = 0.00e+00+0.00e+00j
  max|C_ii^xz(0)|, |C_ii^zx(0)| su 3 siti = 0.00e+00

Totale zeri a t=0 attesi: 24 + 6 = 30


## 6. Il caso $D=0$: conservazione di $M$ e 36 zeri strutturali

In [7]:
J0, b0 = 1.0, 4.0  # punto non degenere a D=0 (b_c=3.0 sarebbe degenere)
H0 = trimer_hamiltonian_dm(J0, b0, 0.0).to_matrix()
E0v, V0 = np.linalg.eigh(H0)
print(f"gap a D=0, b={b0}: {E0v[1]-E0v[0]:.4f}  (non degenere)")

comm_S = np.linalg.norm(Sz_tot @ H0 - H0 @ Sz_tot)
print(f"||[Sz_tot,H]|| a D=0: {comm_S:.2e}  (atteso 0: conservato)\n")

psi0_D0 = V0[:, 0]
imax = np.argmax(np.abs(psi0_D0)); psi0_D0 = psi0_D0 * np.exp(-1j * np.angle(psi0_D0[imax]))
tg = np.linspace(0.05, 6, 30)
predicted = [(i, a, j, b_) for i in (1, 2, 3) for j in (1, 2, 3) for a in paulis for b_ in paulis
             if (a == 'z') + (b_ == 'z') == 1]
observed = [(i, a, j, b_) for i in (1, 2, 3) for j in (1, 2, 3) for a in paulis for b_ in paulis
            if max(abs(classical_exact(i, a, j, b_, t, H0, psi0_D0)) for t in tg) < 1e-9]
print(f"zeri predetti (una sola componente z): {len(predicted)}")
print(f"zeri osservati per ogni t:              {len(observed)}")
print(f"insiemi coincidenti: {set(predicted) == set(observed)}")

gap a D=0, b=4.0: 2.0000  (non degenere)
||[Sz_tot,H]|| a D=0: 0.00e+00  (atteso 0: conservato)



zeri predetti (una sola componente z): 36
zeri osservati per ogni t:              36
insiemi coincidenti: True


## 7. Scan completo delle 81 combinazioni (punto di lavoro con DM, $D=0.15$)

In [8]:
zeri_per_ogni_t = 0
for i in (1, 2, 3):
    for j in (1, 2, 3):
        for a in paulis:
            for b_ in paulis:
                vals = [abs(classical_exact(i, a, j, b_, t, H, psi0)) for t in np.linspace(0.05, 6, 40)]
                if max(vals) < 1e-9:
                    zeri_per_ogni_t += 1
print(f"Combinazioni nulle per ogni t (D={D}): {zeri_per_ogni_t}  (atteso 0, a differenza dei 4 dell'anello)")

zeri_t0 = sum(1 for i in (1, 2, 3) for j in (1, 2, 3) for a in paulis for b_ in paulis
              if abs(classical_exact(i, a, j, b_, 0.0, H, psi0)) < 1e-9)
print(f"Combinazioni nulle a t=0: {zeri_t0}  (atteso 30)")

Combinazioni nulle per ogni t (D=0.15): 0  (atteso 0, a differenza dei 4 dell'anello)
Combinazioni nulle a t=0: 30  (atteso 30)


## 8. Riepilogo

- $P_{13}$ (SWAP puro) commuta con $H$ completo senza bisogno di rotazioni aggiuntive —
  verificato, e verificato che si rompe col segno DM sbagliato.
- $\lambda_\alpha\equiv+1$ per ogni $\alpha$: solo uguaglianze fra correlatori, mai zeri sul
  sito fisso — qualitativamente diverso dall'anello.
- Time-reversal: 24+6=30 zeri a $t=0$, da due argomenti distinti (non uno solo).
- $D=0$: regola di selezione su $M$, 36 zeri per ogni $t$ — verificati coincidere esattamente
  con la predizione.
- A DM acceso ($D=0.15$): 0 zeri per ogni $t\neq0$, 30 zeri a $t=0$ — tutto spiegato.

Prossimo passo: `correlazioni_trimero_catena_esplorazione.ipynb` (circuito con l'ancilla).